# Data Preparation & Augmentation
### Smart Retail Checkout Project

This notebook:
1. Loads the original product images from Google Drive
2. Applies a richer set of augmentations to **increase the dataset size**
3. Builds a **balanced** pool of images per category
4. Splits the data into **Train (70%) / Validation (15%) / Test (15%)**, with an **equal split per category**

Output structure produced:
```
Aug_Dataset/
├── Train/<category>/...
├── Val/<category>/...
└── Test/<category>/...
```


## 1. Setup & Imports

In [ ]:
import os
import cv2
import random
import shutil
import numpy as np

from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array
from tqdm.notebook import tqdm

from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/Smart Retail Checkout'
os.chdir(PROJECT_DIR)

DATASET_PATH  = 'Dataset'          # original images
STAGING_PATH  = 'Staging_Dataset'  # originals + augmented, before splitting
OUTPUT_PATH   = 'Aug_Dataset'      # final Train / Val / Test folders

ITEMS = [
    'Amul_Ice_Cream', 'Cake', 'CeraVe', 'HnS_Shampoo',
    'Nestle_Milk_Powder', 'Plum', 'Thums_Up', 'Wild_Stone'
]

# Every category is augmented up to this many total images (original + augmented),
# so all categories end up perfectly balanced before the split.
TARGET_IMAGES_PER_CLASS = 500

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-6, "Split ratios must sum to 1.0"

## 3. Load Original Images

Loads every image per category and warns if a folder is missing or empty, so a bad path fails fast instead of silently producing an empty dataset later.

In [ ]:
def load_images(dataset_path, items):
    images = {}
    for item in items:
        item_path = os.path.join(dataset_path, item)
        images[item] = []

        if not os.path.exists(item_path):
            print(f"WARNING: Directory not found -> {item_path}")
            continue

        for filename in sorted(os.listdir(item_path)):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(item_path, filename)
                img = cv2.imread(img_path)
                if img is not None:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    images[item].append(img)
                else:
                    print(f"WARNING: Failed to load {img_path}")

        print(f"{item}: {len(images[item])} original images loaded")
    return images


original_images = load_images(DATASET_PATH, ITEMS)

missing = [item for item, imgs in original_images.items() if len(imgs) == 0]
if missing:
    raise ValueError(
        f"No images found for categories: {missing}. "
        f"Check that 'Dataset/<category>' folders exist and contain .jpg/.png files."
    )

## 4. Augmentation Pipeline

A wider augmentation range than before increases visual diversity (rotation, zoom, shift, shear, brightness, and a subtle channel shift). Horizontal/vertical flips stay **off** since these are branded retail products — flipping would mirror logos and text, which is unrealistic and would confuse the classifier.

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=8,
    brightness_range=[0.75, 1.25],
    channel_shift_range=25.0,
    horizontal_flip=False,
    vertical_flip=False,
    fill_mode='nearest'
)

## 5. Build a Balanced Staging Pool

For each category we:
- Copy the original images into a staging folder
- Generate just enough augmented images so every category reaches exactly `TARGET_IMAGES_PER_CLASS` images total

This guarantees class balance *before* the train/val/test split, regardless of how many originals each category started with.

In [ ]:
def prepare_staging_dirs(staging_path, items):
    if os.path.exists(staging_path):
        shutil.rmtree(staging_path)
    for item in items:
        os.makedirs(os.path.join(staging_path, item), exist_ok=True)


def save_original_images(images, staging_path):
    for item, imgs in images.items():
        item_dir = os.path.join(staging_path, item)
        for idx, img in enumerate(imgs):
            out_path = os.path.join(item_dir, f"{item}_orig_{idx:04d}.jpg")
            cv2.imwrite(out_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))


def augment_class(item, imgs, staging_path, target_total, datagen):
    """Generate just enough augmented images so this class reaches target_total
    (originals + augmented), spreading the augmentation count evenly across
    the available source images."""
    item_dir = os.path.join(staging_path, item)
    n_original = len(imgs)
    if n_original == 0:
        return 0

    n_to_generate = max(0, target_total - n_original)
    if n_to_generate == 0:
        return 0

    per_image = n_to_generate // n_original
    remainder = n_to_generate % n_original

    generated = 0
    for idx, img in enumerate(imgs):
        n_for_this_image = per_image + (1 if idx < remainder else 0)
        if n_for_this_image == 0:
            continue

        x = img_to_array(img)
        x = x.reshape((1,) + x.shape)

        i = 0
        for batch in datagen.flow(
            x, batch_size=1,
            save_to_dir=item_dir,
            save_prefix=f"{item}_aug_{idx:04d}",
            save_format='jpg'
        ):
            i += 1
            generated += 1
            if i >= n_for_this_image:
                break

    return generated


prepare_staging_dirs(STAGING_PATH, ITEMS)
save_original_images(original_images, STAGING_PATH)

print("Generating augmented images...\n")
for item in tqdm(ITEMS, desc="Augmenting classes"):
    n_gen = augment_class(item, original_images[item], STAGING_PATH, TARGET_IMAGES_PER_CLASS, datagen)
    total_now = len([f for f in os.listdir(os.path.join(STAGING_PATH, item))
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"{item}: +{n_gen} augmented -> {total_now} total images")

## 6. Split into Train / Validation / Test (70 / 15 / 15)

Each category is shuffled and split **independently**, then capped to the same `TARGET_IMAGES_PER_CLASS` size — so every category contributes an equal number of images to Train, Val, and Test (a proper stratified split).

In [ ]:
def split_dataset(staging_path, output_path, items, train_ratio, val_ratio, test_ratio,
                   target_per_class, seed=42):
    if os.path.exists(output_path):
        shutil.rmtree(output_path)

    split_names = ['Train', 'Val', 'Test']
    for split in split_names:
        for item in items:
            os.makedirs(os.path.join(output_path, split, item), exist_ok=True)

    rng = random.Random(seed)
    summary = {}

    for item in items:
        item_dir = os.path.join(staging_path, item)
        all_files = [f for f in os.listdir(item_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        rng.shuffle(all_files)

        # Cap so every category contributes an EQUAL number of images to the split
        all_files = all_files[:target_per_class]
        n_total = len(all_files)

        n_train = int(round(n_total * train_ratio))
        n_val = int(round(n_total * val_ratio))
        n_test = n_total - n_train - n_val  # remainder absorbed by test to avoid rounding loss

        train_files = all_files[:n_train]
        val_files = all_files[n_train:n_train + n_val]
        test_files = all_files[n_train + n_val:]

        for split, files in zip(split_names, [train_files, val_files, test_files]):
            for f in files:
                src = os.path.join(item_dir, f)
                dst = os.path.join(output_path, split, item, f)
                shutil.copy2(src, dst)

        summary[item] = {
            'total': n_total,
            'train': len(train_files),
            'val': len(val_files),
            'test': len(test_files)
        }

    return summary


summary = split_dataset(
    STAGING_PATH, OUTPUT_PATH, ITEMS,
    TRAIN_RATIO, VAL_RATIO, TEST_RATIO,
    TARGET_IMAGES_PER_CLASS, seed=RANDOM_SEED
)

## 7. Verify the Split

In [ ]:
print(f"{'Category':<22}{'Total':>8}{'Train':>8}{'Val':>8}{'Test':>8}")
print("-" * 54)
for item, counts in summary.items():
    print(f"{item:<22}{counts['total']:>8}{counts['train']:>8}{counts['val']:>8}{counts['test']:>8}")

total_train = sum(c['train'] for c in summary.values())
total_val = sum(c['val'] for c in summary.values())
total_test = sum(c['test'] for c in summary.values())
grand_total = total_train + total_val + total_test

print("-" * 54)
print(f"{'TOTAL':<22}{grand_total:>8}{total_train:>8}{total_val:>8}{total_test:>8}")
print(f"\nActual split ratio -> Train: {total_train/grand_total:.1%}  "
      f"Val: {total_val/grand_total:.1%}  Test: {total_test/grand_total:.1%}")

## 8. (Optional) Clean Up Staging Folder

The staging folder just duplicates what's now in `Aug_Dataset/`. Uncomment to free up Drive space once you're happy with the result.

In [ ]:
shutil.rmtree(STAGING_PATH)
print("Staging folder removed.")